[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/16_retries.ipynb)

# 16. Retries and Recovery

**Part 4: Reliability** · about 25 minutes

Every notebook so far has assumed the model API answers every request. In real use it doesn't. You'll see rate limits (`429`), overloaded servers (`503`), requests that hang, and connections that drop. For a single chat message that's an annoyance. For an agent that's 30 steps into a task, one unhandled error throws all of that work away.

This notebook makes the agent survive that. To test failures on demand, we build a small **flaky proxy**: a server that sits between the agent and the model and breaks requests in whatever way we tell it to. It works the same in offline mode and with a real provider.

**By the end you will have:**

- a retry function that knows which errors to retry and how long to wait
- watched it ride out a rate limit, a server error, a hang and a dropped connection
- checkpoints, so an agent can resume a task after a long outage
- handled the quiet failure: a response cut off by `max_tokens`

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import json
import random
import tempfile
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import requests

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"

## 1. A proxy that breaks things on purpose

The proxy forwards every request to the real `llm.base_url`, unless its **plan** says to fail this one. The plan is a list we control, consumed one request at a time:

| Plan entry | What the client sees |
|---|---|
| `429` | "Too many requests", with a `Retry-After` header |
| `500`, `503` | A server error |
| `"hang"` | No answer at all, until the client gives up |
| `"drop"` | The connection closes with no response |
| `"ok"` or an empty plan | The real response |

Chaos testing like this is how reliability work is done for real. You don't wait for an outage to find out what your code does in one.

In [3]:
class FlakyProxy:
    def __init__(self, upstream):
        self.upstream = upstream.rstrip("/")
        self.plan, self.log = [], []
        proxy = self

        class Handler(BaseHTTPRequestHandler):
            def log_message(self, *args):
                pass

            def do_POST(self):
                body = self.rfile.read(int(self.headers.get("Content-Length", 0)))
                action = proxy.plan.pop(0) if proxy.plan else "ok"
                proxy.log.append(action)
                try:
                    if action == "hang":
                        time.sleep(3)  # longer than the client's timeout below
                        return
                    if action == "drop":
                        self.connection.close()
                        return
                    if isinstance(action, int):
                        error = json.dumps({"error": {"message": f"simulated {action}"}}).encode()
                        self.send_response(action)
                        if action == 429:
                            self.send_header("Retry-After", "1")
                        self.send_header("Content-Type", "application/json")
                        self.send_header("Content-Length", str(len(error)))
                        self.end_headers()
                        self.wfile.write(error)
                        return
                    forward = {k: v for k, v in self.headers.items() if k.lower() in ("authorization", "content-type")}
                    reply = requests.post(proxy.upstream + self.path.removeprefix("/v1"), data=body,
                                          headers=forward, timeout=llm.timeout)
                    self.send_response(reply.status_code)
                    self.send_header("Content-Type", "application/json")
                    self.send_header("Content-Length", str(len(reply.content)))
                    self.end_headers()
                    self.wfile.write(reply.content)
                except (BrokenPipeError, ConnectionResetError, OSError):
                    pass  # the client gave up first, which is what we wanted to test

        class QuietServer(ThreadingHTTPServer):
            def handle_error(self, request, client_address):
                pass  # dropped connections are the point here, so don't print tracebacks

        self.server = QuietServer(("127.0.0.1", 0), Handler)
        threading.Thread(target=self.server.serve_forever, daemon=True).start()
        self.url = f"http://127.0.0.1:{self.server.server_address[1]}/v1/chat/completions"


proxy = FlakyProxy(llm.base_url)
PAYLOAD = {"model": llm.model, "messages": [{"role": "user", "content": "In one sentence, what is a large language model?"}]}

proxy.plan = [429]
naive = requests.post(proxy.url, headers=headers, json=PAYLOAD, timeout=10)
print("A plain request during a rate limit:", naive.status_code, naive.text)

A plain request during a rate limit: 429 {"error": {"message": "simulated 429"}}


That's how every agent in this course would have died. One `429` and `raise_for_status()` ends the run.

## 2. What to retry, and how long to wait

Not every error deserves a retry. Retrying a `400` just sends the same broken request again. The rules:

| Error | Retry? | Why |
|---|---|---|
| `429` rate limit | Yes, after `Retry-After` if given | You're sending too fast, so wait |
| `500`, `502`, `503`, `504` | Yes | A temporary problem on their side |
| Timeout, dropped connection | Yes | Usually a network blip |
| `400`, `401`, `403`, `404` | **No** | Your request or your key is wrong. Retrying changes nothing. |

And between attempts, **exponential backoff with jitter**: wait 0.5s, then 1s, then 2s and so on, each multiplied by a random factor. The doubling gives a struggling service room to recover. The randomness stops a thousand clients that failed at the same moment from all retrying at the same moment too, which would knock the service over again.

In [4]:
RETRYABLE = {408, 409, 429, 500, 502, 503, 504}


class GaveUp(Exception):
    pass


DEMO_TIMEOUT = 2.0 if llm.offline else llm.timeout  # short offline so the demo is quick; real models need longer


def post_with_retries(url, payload, max_attempts=6, base_delay=0.5, max_delay=8.0, timeout=DEMO_TIMEOUT, verbose=True):
    """POST with retries for temporary failures. Raises at once for errors a retry can't fix."""
    for attempt in range(1, max_attempts + 1):
        retry_after = None
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=timeout)
        except requests.Timeout:
            problem = "timed out"
        except requests.ConnectionError:
            problem = "connection failed"
        else:
            if response.status_code < 400:
                return response.json()
            if response.status_code not in RETRYABLE:
                raise RuntimeError(f"HTTP {response.status_code}, not retrying: {response.text[:200]}")
            problem = f"HTTP {response.status_code}"
            retry_after = response.headers.get("Retry-After")

        if attempt == max_attempts:
            raise GaveUp(f"gave up after {attempt} attempts, last problem: {problem}")
        if retry_after:
            delay = float(retry_after)
        else:
            delay = min(max_delay, base_delay * 2 ** (attempt - 1)) * random.uniform(0.5, 1.0)
        if verbose:
            print(f"  attempt {attempt}: {problem}, waiting {delay:.2f}s")
        time.sleep(delay)

In offline mode the timeout is 2 seconds so the demo is quick. With a real model it's `llm.timeout`, because a slow answer is not a failure. We learned that the hard way: the first version of this notebook used 2 seconds everywhere, and on a local model every single attempt timed out, so the retry logic gave up on a service that was working fine. A timeout that's too short turns every long answer into a retry, and each retry costs money.

## 3. Riding out a bad few seconds

Four different failures in a row, then success.

In [5]:
proxy.plan = [429, 503, "hang", "drop"]
proxy.log.clear()

started = time.perf_counter()
data = post_with_retries(proxy.url, PAYLOAD)
print(f"\nSucceeded after {len(proxy.log)} attempts in {time.perf_counter() - started:.1f}s")
print("Answer:", data["choices"][0]["message"]["content"])
print("What the proxy did:", proxy.log)

  attempt 1: HTTP 429, waiting 1.00s


  attempt 2: HTTP 503, waiting 0.53s


  attempt 3: timed out, waiting 1.35s


  attempt 4: connection failed, waiting 2.45s



Succeeded after 5 attempts in 10.3s
Answer: A large language model is a neural network trained on huge amounts of text to predict what comes next, which lets it generate useful language.
What the proxy did: [429, 503, 'hang', 'drop', 'ok']


In [6]:
# Checkpoint: four failures absorbed, the fifth attempt got through.
assert proxy.log == [429, 503, "hang", "drop", "ok"]
print("Checkpoint passed.")

Checkpoint passed.


## 4. Knowing when to stop

Retrying forever is its own bug. A request that can't succeed should fail quickly, and an outage should end in a clear error, not an agent stuck in a loop.

In [7]:
proxy.plan = [400]
try:
    post_with_retries(proxy.url, PAYLOAD)
except RuntimeError as error:
    print("400:", error)
print("Attempts made:", len(proxy.log) - 5)

proxy.plan = [503] * 10
started = time.perf_counter()
try:
    post_with_retries(proxy.url, PAYLOAD, max_attempts=4)
except GaveUp as error:
    print(f"\nLong outage: {error} ({time.perf_counter() - started:.1f}s)")

400: HTTP 400, not retrying: {"error": {"message": "simulated 400"}}
Attempts made: 1
  attempt 1: HTTP 503, waiting 0.25s


  attempt 2: HTTP 503, waiting 0.60s


  attempt 3: HTTP 503, waiting 1.41s



Long outage: gave up after 4 attempts, last problem: HTTP 503 (2.6s)


In [8]:
# Checkpoint: the 400 got exactly one attempt; the outage got exactly four.
assert proxy.log[5:] == [400, 503, 503, 503, 503]
proxy.plan.clear()
print("Checkpoint passed.")

Checkpoint passed.


## 5. Checkpoints: surviving a long outage

Retries handle seconds of trouble. For an outage of minutes or hours, the right move is to **stop cleanly and resume later**. Because the message list is the agent's entire state (notebook 01), that's simple: save it to disk after every step, and a new run can pick up where the old one stopped.

We'll use the research agent from notebook 04. The proxy lets the first two model calls through, then fails every request.

In [9]:
NOTES = {
    "standup.md": "Standup notes. The launch deadline moved to March 14 because of the audit.",
    "owners.md": "Launch owner: Priya. Backend owner: Sam.",
    "lunch.md": "The team voted for dosa on Friday.",
}
TOOLS = [
    {"type": "function", "function": {"name": "list_files", "description": "List the names of all note files.",
                                      "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {"name": "read_file", "description": "Read one note file by its exact name.",
                                      "parameters": {"type": "object", "properties": {"path": {"type": "string"}},
                                                     "required": ["path"]}}},
]
checkpoint = Path(tempfile.mkdtemp(prefix="agent-checkpoints-")) / "run.json"


def run_agent(question=None, resume=False, max_steps=10):
    if resume:
        messages = json.loads(checkpoint.read_text())
        print(f"Resuming from a checkpoint with {len(messages)} messages.")
    else:
        messages = [
            {"role": "system", "content": "Answer using the tools. Read the files you need, then reply briefly."},
            {"role": "user", "content": question},
        ]
    for step in range(1, max_steps + 1):
        data = post_with_retries(proxy.url, {"model": llm.model, "messages": messages, "tools": TOOLS},
                                 max_attempts=3, timeout=llm.timeout, verbose=False)
        message = data["choices"][0]["message"]
        messages.append(message)
        if message.get("tool_calls"):
            for call in message["tool_calls"]:
                path = json.loads(call["function"]["arguments"] or "{}").get("path")
                result = json.dumps(sorted(NOTES)) if call["function"]["name"] == "list_files" \
                    else NOTES.get(path, f"Error: no file named {path!r}")
                messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
                print(f"  step {step}: {call['function']['name']}({call['function']['arguments']})")
        checkpoint.write_text(json.dumps(messages))  # after the step is complete, never halfway through
        if not message.get("tool_calls"):
            return message.get("content")


proxy.plan = ["ok", "ok"] + [503] * 3
try:
    run_agent("What is the launch deadline, and who owns the launch?")
except GaveUp as error:
    print(f"\nThe run stopped: {error}")
    print(f"Saved state: {len(json.loads(checkpoint.read_text()))} messages")

  step 1: list_files({})


  step 2: read_file({"path": "lunch.md"})



The run stopped: gave up after 3 attempts, last problem: HTTP 503
Saved state: 6 messages


The service is down, the run has stopped, and nothing is lost. Some time later, the service recovers:

In [10]:
proxy.plan.clear()
answer = run_agent(resume=True)
print("\nAnswer:", answer)

Resuming from a checkpoint with 6 messages.


  step 1: read_file({"path": "owners.md"})
  step 1: read_file({"path": "standup.md"})



Answer: The launch deadline is March 14, and the owner is Priya.


In [11]:
# Checkpoint: the resumed run finished the task without redoing the steps before the outage.
if llm.offline:
    assert "March 14" in answer and "Priya" in answer
print("Checkpoint passed.")

Checkpoint passed.


The checkpoint is written only **after a step is complete**, meaning after every tool result for that step has been added. If you saved halfway, a resumed run would start with an assistant message whose tool calls have no results, and the API would reject it, or the agent would run the same tool twice.

That second risk matters beyond checkpoints. **Retrying a model call is safe**, because asking again changes nothing in the world. **Retrying a tool call may not be**. Running tests twice is harmless, but sending an email or charging a card twice isn't. Before you add retries to a tool, ask whether doing it twice is the same as doing it once. If not, the tool needs a way to tell whether the first attempt already happened.

## 6. The quiet failure: a cut-off answer

Some failures come back as HTTP 200. If a response hits the `max_tokens` limit, the provider stops generating mid-sentence and reports `finish_reason: "length"`. For a chat answer you get a truncated reply. For a **tool call** you can get half a JSON object, and an agent that doesn't check will try to parse it and crash, or pass along garbage.

We can't easily make every model run out of tokens on cue, so here's the check, tested against a hand-made response:

In [12]:
def check_finish(data):
    """Return the message, or raise if it was cut off before it finished."""
    choice = data["choices"][0]
    if choice.get("finish_reason") == "length":
        raise ValueError("The response hit max_tokens and was cut off. "
                         "Raise max_tokens, or ask for a shorter answer, then retry this step.")
    return choice["message"]


cut_off = {"choices": [{"finish_reason": "length", "message": {
    "role": "assistant", "content": None,
    "tool_calls": [{"id": "x", "type": "function",
                    "function": {"name": "write_file", "arguments": '{"path": "cart.py", "content": "def sub'}}]}}]}

try:
    check_finish(cut_off)
except ValueError as error:
    print("Caught:", error)

Caught: The response hit max_tokens and was cut off. Raise max_tokens, or ask for a shorter answer, then retry this step.


Without this check, notebook 05's agent would have tried `json.loads` on `{"path": "cart.py", "content": "def sub`, and the dispatcher would have returned a confusing JSON error to the model. The fix depends on the cause. Either the limit is too low for the job, or the model is trying to write something huge in one go, like a whole file, which notebook 06's precise edits avoid.

## Recap

- Retry `429`, `5xx`, timeouts and dropped connections. Never retry `4xx` errors you caused.
- Back off exponentially with jitter, honour `Retry-After`, and give up after a fixed number of attempts.
- Checkpoint the message list after each **complete** step, so long outages cost a pause, not the whole task.
- Model calls are safe to retry. Tool calls are only safe if doing them twice is harmless.
- Check `finish_reason`. A `200` can still be a broken answer.

## Exercises

1. Make the agent resume automatically: catch `GaveUp`, wait 30 seconds, and try again from the checkpoint, at most three times.
2. Add a **circuit breaker**: after five failures in a row across all requests, stop trying for a minute, failing at once instead. Why does this help a service that's struggling?
3. Count how much time each run spends waiting in retries, and print it with the result. In a real run, where does the time go?

## Clean up

In [13]:
proxy.server.shutdown()
checkpoint.unlink(missing_ok=True)
print("Stopped the proxy and removed the checkpoint.")

Stopped the proxy and removed the checkpoint.


**Next:** [17. Tracing](17_tracing.ipynb). Recording everything an agent does, so you can see why a run went wrong.